# Notebook 3: AlphaGenome and TP53 mutations in CCLE

**LLMs in Bioinformatics workshop, AlphaGenome block** | ~60 minutes | internet and a free API key needed | no GPU

**AlphaGenome** (Google DeepMind) reads up to 1 million bases of DNA and predicts thousands of molecular readouts: gene expression, splicing, chromatin accessibility. You do not run the model; you **call it through a web API** and get tracks back.

In Notebook 2 you retrieved the layers of the CCLE cancer cell lines. Here you use the **DNA layer**: the somatic mutations of the cell lines, with their genomic coordinates. You will ask AlphaGenome what each mutation should do to the **TP53 gene's RNA**, and then compare with what the cell lines that carry the mutation actually show in the **RNA layer**.

A prediction you can reason about in advance: **a mutation that breaks a splice site, or creates a stop codon, should lower the RNA of the gene** (the cell destroys faulty transcripts), while a **silent** change should do nothing. A model that reads DNA sequence may see the first (splicing) and may not see the second (the cell's quality control). That is the lesson of the notebook.

By the end you will be able to:

1. connect to the AlphaGenome API and predict RNA-seq for a gene locus;
2. build **variants correctly from a data table** (right position, right reference base, right genome build);
3. **score variants** and read the tidy results table;
4. compare **predicted effects with measured RNA** and say what the model can and cannot see;
5. name three ways a variant-effect pipeline silently goes wrong: **genome build, strand, reference allele**.

### Time plan (60 min)

| Time | Part | What happens |
| --- | --- | --- |
| 0:00-0:08 | **0. Setup** | install, API key, terms of use, load the CCLE tables |
| 0:08-0:18 | **1. The TP53 locus** | predicted expression track for the whole gene |
| 0:18-0:38 | **2. Variants from the CCLE table** | build variants, check the reference bases |
| 0:38-0:54 | **3. Score and compare** | predicted effect vs measured RNA of the carrying cell lines |
| 0:54-1:00 | **4. Pitfalls and open time** | genome build, limits |

Each part: **predict -> try (TODO) -> check cell -> debrief**.

## Part 0. Setup (8 min)

**Get a key (before the session).** Request a free API key at <https://alphagenome.google/api>. In Colab, open the key icon in the left sidebar (**Secrets**), add a secret named `ALPHA_GENOME_API_KEY` (with underscores exactly as written), and switch on **Notebook access**. If you skip this, the notebook asks you to paste the key instead. Never save a notebook with your key typed into a cell.

**Terms of use (read them).** The API is free for **non-commercial** use. Per the terms, outputs **must not be used to train other machine-learning models** and **must not be used for clinical decisions**. Query rates vary with demand, so keep requests small.

**Data.** Upload two files from your instructor to this Colab session (left sidebar, Files): `ccle_mutations.csv` (the somatic mutations of the cell lines, **genome build hg38**) and `ccle_panel.csv` (the RNA level per cell line and gene).

In [ ]:
!pip -q install alphagenome pyliftover requests 2>&1 | tail -1

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

from alphagenome import colab_utils
from alphagenome.data import gene_annotation, genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.models.v1 import dna_client, variant_scorers
from alphagenome.visualization import plot_components

def get_key():
    """Colab secret first (name ALPHA_GENOME_API_KEY); otherwise ask for the key without echoing it."""
    try:
        return colab_utils.get_api_key()
    except Exception:
        import getpass
        print("No Colab secret found (or notebook access is off). Paste your AlphaGenome API key below.")
        key = getpass.getpass("AlphaGenome API key: ").strip()
        if not key:
            raise RuntimeError("No API key. Request one at https://alphagenome.google/api, then re-run this cell.")
        return key

def connect(key):
    model = dna_client.create(key)
    # create() does not check the key, so make one tiny request now to find out
    try:
        model.predict_sequence(sequence="ACGT" * 4096, requested_outputs=[dna_client.OutputType.RNA_SEQ], ontology_terms=None)
    except Exception as e:
        if "API key not valid" in str(e) or "INVALID_ARGUMENT" in str(e):
            raise ValueError("AlphaGenome says this API key is not valid. Check: (1) you copied the whole key, with no spaces or quote marks; "
                             "(2) the Colab secret VALUE is the key itself, not the secret name; (3) the key was issued at https://alphagenome.google/api. "
                             "Fix the secret (or paste the key again) and re-run this cell.") from None
        print("note: the test request failed for another reason, continuing:", type(e).__name__)
    return model

dna_model = connect(get_key().strip().strip("'\""))
print("connected to AlphaGenome")
print("supported sequence lengths:", list(dna_client.SUPPORTED_SEQUENCE_LENGTHS.keys()))

## Part 1. The TP53 locus (10 min)

We need **gene annotation**: where TP53 and its exons are in the human genome (build **hg38**). DeepMind provides the GENCODE annotation as a single file.

In [ ]:
GENE = "TP53"
gtf = pd.read_feather("https://storage.googleapis.com/alphagenome/reference/gencode/hg38/gencode.v46.annotation.gtf.gz.feather")
gtf_mane = gene_annotation.filter_to_mane_select_transcript(gene_annotation.filter_protein_coding(gtf))
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_mane)

gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=GENE)
print(gene_interval)
print("gene length:", gene_interval.width, "bp; strand:", gene_interval.strand)

**Prediction:** TP53 is on chromosome 17. Is it on the `+` or the `-` strand? (The printout above tells you; write your guess first next time.) Why does the strand matter for everything that follows?

Now predict RNA-seq for the locus in one tissue. The ontology term `UBERON:0001114` is "right liver lobe" (used in the AlphaGenome tutorial). The model needs one of its supported input lengths, so we resize the gene interval to 1 Mb.

In [ ]:
TISSUE = ["UBERON:0001114"]       # right liver lobe
interval_1mb = gene_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

t0 = time.time()
out = dna_model.predict_interval(
    interval=interval_1mb,
    requested_outputs=[dna_client.OutputType.RNA_SEQ],
    ontology_terms=TISSUE,
)
print(f"prediction took {time.time() - t0:.0f} s; RNA-seq tracks: {out.rna_seq.values.shape}")

transcripts = transcript_extractor.extract(interval_1mb)
plot_components.plot(
    components=[plot_components.TranscriptAnnotation(transcripts), plot_components.Tracks(out.rna_seq)],
    interval=gene_interval.resize(2**17),
)
plt.show()

**Debrief.** Where does the predicted signal sit relative to the exons? Which strand track carries the signal, and does that agree with your answer about the TP53 strand? A prediction here is **not an experiment**: it is a model's reading of the sequence.

## Part 2. Variants from the CCLE table (20 min)

The table `ccle_mutations.csv` has one row per mutation per cell line, with the columns `cell_line`, `gene`, `chrom`, `pos`, `ref`, `alt` and `variant_info` (the type of mutation). Coordinates are **1-based, hg38**.

In [ ]:
# Class data files: downloaded automatically from the course site (nothing to upload).
import os, requests
CLASS_FILES_URL = "https://raw.githubusercontent.com/AmirAliLoghmani80/CBH-Workshop/main/data/"
def get_class_file(name):
    if os.path.exists(name):
        return name
    if "YOUR-USER" in CLASS_FILES_URL:
        print(f"NOTE: {name} not found and no download address is set. Upload it (left sidebar, Files) or ask the instructor.")
        return None
    r = requests.get(CLASS_FILES_URL + name, timeout=120)
    if r.status_code != 200 or r.content[:15].lstrip().lower().startswith((b"<!doctype", b"<html")):
        print(f"NOTE: could not download {name} (HTTP {r.status_code}). Upload it (left sidebar, Files) or ask the instructor.")
        return None
    with open(name, "wb") as f:
        f.write(r.content)
    print(f"downloaded {name} ({len(r.content)/1e6:.1f} MB)")
    return name

get_class_file("ccle_mutations.csv"); get_class_file("ccle_panel.csv")
muts_all = pd.read_csv("ccle_mutations.csv")
need = {"cell_line", "gene", "chrom", "pos", "ref", "alt", "variant_info"}
if need - set(muts_all.columns):
    raise ValueError(f"ccle_mutations.csv lacks the columns {sorted(need - set(muts_all.columns))}; found {list(muts_all.columns)}. Ask the instructor for the class file.")
muts = muts_all[muts_all["gene"] == GENE].copy()
print(f"{len(muts)} mutation records of {GENE} in {muts['cell_line'].nunique()} cell lines")
print(muts["variant_info"].value_counts().head(12))

We keep **single-base substitutions** (SNVs): one base replaced by one base. Insertions and deletions need a different representation, so we leave them out here (a real analysis would not). And we group the mutation types into classes:

| Class | What it is | Expected effect on RNA |
| --- | --- | --- |
| `splice` | at the first or last two bases of an intron (`splice_donor_variant`, `splice_acceptor_variant`) | splicing breaks, transcript often degraded: **lower** |
| `nonsense` | creates a stop codon (`stop_gained`) | the cell destroys the message (nonsense-mediated decay): **lower** |
| `missense` | changes one amino acid | the RNA is normal: **no change** |
| `silent` | changes the DNA, not the amino acid (`synonymous_variant`) | **no change** |
| `silent_near_splice` | a synonymous change that also touches the edge of a splice site (`splice_region_variant&synonymous_variant`) | not sure: that is the question |

In [ ]:
def variant_class(info):
    s = str(info).lower()          # DepMap writes Ensembl VEP terms, e.g. "missense_variant&splice_region_variant"
    if "splice_donor" in s or "splice_acceptor" in s or ("splice" in s and "region" not in s): return "splice"
    if "stop_gained" in s or "nonsense" in s: return "nonsense"
    if "missense" in s: return "missense"
    if "synonymous" in s or "silent" in s:
        return "silent_near_splice" if "splice_region" in s else "silent"      # a silent change next to a splice site is not really silent
    return "other"                 # includes splice_region, frameshift, inframe, stop_lost

CLASSES = ["splice", "nonsense", "missense", "silent", "silent_near_splice"]
is_snv = muts["ref"].isin(list("ACGT")) & muts["alt"].isin(list("ACGT"))
muts["cls"] = muts["variant_info"].map(variant_class)
snv = muts[is_snv & muts["cls"].isin(CLASSES)].copy()
print(f"{is_snv.sum()} of {len(muts)} records are single-base substitutions; {len(snv)} fall in the classes")
print(snv["cls"].value_counts())

**TODO 1.** Write `to_variant(chrom, pos, ref, alt, name)` returning an AlphaGenome `genome.Variant`. AlphaGenome wants the chromosome written `chr17` (with `chr`), the position as an **integer** (1-based), and upper-case bases. Tables write chromosomes in different ways (`17`, `chr17`): handle both.

In [ ]:
def to_variant(chrom, pos, ref, alt, name):
    chrom = str(chrom)
    # TODO 1a: make sure the chromosome starts with "chr" (without doubling it)
    chrom = ...
    # TODO 1b: build the Variant with an integer position and upper-case bases
    return ...

In [ ]:
# CHECK 1
try:
    v = to_variant("17", 100, "a", "G", "t")
    assert v.chromosome == "chr17", f"chromosome should be 'chr17', got {v.chromosome!r}"
    assert to_variant("chr17", 100, "A", "G", "t").chromosome == "chr17", "'chr17' must not become 'chrchr17'"
    assert v.position == 100 and isinstance(v.position, int), "position must be an int"
    assert v.reference_bases == "A", "bases must be upper case"
    print("OK. to_variant works")
except (TypeError, AttributeError) as e:
    raise AssertionError("TODO 1 not finished: build genome.Variant(chromosome=..., position=int(pos), reference_bases=..., alternate_bases=..., name=...)") from e

### Reference bases

AlphaGenome does **not** stop you if you give the wrong reference base. If your position is off by one, or comes from another genome build, you silently score the wrong variant. So we **look up the real base** in the reference genome (Ensembl REST service, with UCSC as a backup; both GRCh38 / hg38) and compare it with the `ref` column of the table.

In [ ]:
_ref_cache = {}

def _ensembl(c, pos):
    r = requests.get(f"https://rest.ensembl.org/sequence/region/human/{c}:{pos}..{pos}:1?coord_system_version=GRCh38",
                     headers={"Content-Type": "text/plain"}, timeout=60)
    r.raise_for_status()
    return r.text.strip().upper()

def _ucsc(c, pos):
    r = requests.get(f"https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom=chr{c};start={pos - 1};end={pos}", timeout=60)
    r.raise_for_status()
    return r.json()["dna"].strip().upper()

def ref_base(chrom, pos):
    """Reference base at a 1-based GRCh38 position. Tries Ensembl (with retries), then UCSC."""
    c = chrom.replace("chr", "")
    key = (c, int(pos))
    if key in _ref_cache:
        return _ref_cache[key]
    errors = []
    for name, fn in [("Ensembl", _ensembl), ("Ensembl", _ensembl), ("UCSC", _ucsc), ("UCSC", _ucsc)]:
        try:
            base = fn(c, int(pos))
            if base in ("A", "C", "G", "T"):
                _ref_cache[key] = base
                time.sleep(0.1)
                return base
            errors.append(f"{name}: unexpected answer {base!r}")
        except Exception as e:
            errors.append(f"{name}: {type(e).__name__}")
            time.sleep(1.5)
    raise RuntimeError(f"Could not look up the reference base at {chrom}:{pos}. Tried {errors}. "
                       "Wait a minute and re-run this cell (these public services sometimes fail briefly).")

We choose a small, balanced set: **up to `N_PER_CLASS` distinct variants per class**, preferring those carried by cell lines that have RNA data. Each distinct variant is scored once, even if several cell lines carry it.

In [ ]:
N_PER_CLASS = 15
panel = pd.read_csv("ccle_panel.csv")
rna = panel.pivot_table(index="cell_line", columns="gene", values="rna_log2tpm")
if GENE not in rna.columns:
    raise ValueError(f"{GENE} is not in ccle_panel.csv")
has_rna = set(rna.index[rna[GENE].notna()])

snv["key"] = snv["chrom"].astype(str).str.replace("chr", "", regex=False) + ":" + snv["pos"].astype(int).astype(str) + ":" + snv["ref"] + ">" + snv["alt"]
snv["has_rna"] = snv["cell_line"].isin(has_rna)
distinct = (snv.groupby(["key", "cls"]).agg(chrom=("chrom", "first"), pos=("pos", "first"), ref=("ref", "first"), alt=("alt", "first"),
                                           carriers=("cell_line", lambda s: list(s)), n_rna=("has_rna", "sum")).reset_index())
distinct = distinct[distinct["n_rna"] > 0]
rng = np.random.default_rng(0)
picked = []
for cls in CLASSES:
    d = distinct[distinct["cls"] == cls]
    picked.append(d.sample(min(N_PER_CLASS, len(d)), random_state=0) if len(d) else d)
picked = pd.concat(picked).reset_index(drop=True)
print(picked["cls"].value_counts().to_string())

variants = [to_variant(r.chrom, r.pos, r.ref, r.alt, f"{r.cls}_{i}") for i, r in picked.iterrows()]

# the reference check: the base in the genome must equal the table's ref
t0 = time.time()
genome_ref = [ref_base(v.chromosome, v.position) for v in variants]
match = np.array([g == v.reference_bases for g, v in zip(genome_ref, variants)])
print(f"reference base matches the genome for {match.mean():.0%} of {len(variants)} variants ({time.time() - t0:.0f} s)")
assert match.mean() >= 0.9, "Most reference bases do not match hg38: wrong genome build, or positions shifted by one. Do not score these."
if not match.all():
    print("mismatching variants (excluded):", [v.name for v, m in zip(variants, match) if not m])
picked = picked[match].reset_index(drop=True)
variants = [v for v, m in zip(variants, match) if m]

**Why this check matters.** A table from an older paper in **hg19** would fail this test for almost every row. A table that uses 0-based positions would fail for about 3 in 4 rows. Without the check, AlphaGenome would happily score the wrong bases.

**Strand.** The table gives `ref` and `alt` on the **forward (+) strand of the genome**, whatever the strand of the gene. TP53 is read from the `-` strand, so a `C>T` in the table is a `G>A` in the gene's own reading direction. AlphaGenome takes forward-strand variants and handles the gene's strand itself, but when you read about a mutation as "G>A" in a paper, ask which strand it was written on.

## Part 3. Score and compare (16 min)

`score_variants` predicts the reference and the alternate allele for each variant and condenses the difference into a score, using a **variant scorer**. AlphaGenome ships recommended scorers. We use two: `SPLICE_SITES` (change in predicted splice-site strength) and `RNA_SEQ` (change in expression, per gene, in each of many tracks).

**Prediction:** rank the classes by the **drop in predicted TP53 expression** they cause, from largest drop to none: splice, nonsense, missense, silent, and silent-near-splice (where does that one go?). And: which classes do you expect AlphaGenome to be able to see at all? Write it down.

**My ranking and reasoning:** ____________________

In [ ]:
scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS["SPLICE_SITES"],
           variant_scorers.RECOMMENDED_VARIANT_SCORERS["RNA_SEQ"]]
CONTEXT = dna_client.SEQUENCE_LENGTH_100KB      # 131 kb of context around each variant: faster than 1 Mb

t0 = time.time()
intervals = [v.reference_interval.resize(CONTEXT) for v in variants]
scores = dna_model.score_variants(intervals=intervals, variants=variants, variant_scorers=scorers)
print(f"scored {len(variants)} variants in {time.time() - t0:.0f} s")
tidy = variant_scorers.tidy_scores(scores, match_gene_strand=True)
# depending on the client version, variant_id is a string or a Variant object (not hashable); make it a plain string either way
tidy["variant_id"] = tidy["variant_id"].map(lambda v: getattr(v, "name", None) or str(v))
print(tidy.shape)
tidy[["variant_id", "gene_name", "output_type", "track_name", "raw_score"]].head()

`tidy` has one row per variant, gene and track (thousands of tracks across many tissues). To compare variants we need **one number per variant**:

- **splice score**: for `SPLICE_SITES`, the largest absolute raw score over all tracks;
- **expression change**: for `RNA_SEQ`, the **mean raw score over all tracks** for TP53 (negative means lower predicted expression).

**TODO 2.** Finish `per_variant_expression`: a Series indexed by `variant_id` with the **mean** of `raw_score` over the `RNA_SEQ` rows of the gene.

In [ ]:
def per_variant_splice(tidy, gene=GENE):
    d = tidy[(tidy["gene_name"] == gene) & (tidy["output_type"].astype(str).str.contains("SPLICE_SITES"))].copy()
    d["abs_score"] = d["raw_score"].abs()
    return d.groupby("variant_id")["abs_score"].max()

def per_variant_expression(tidy, gene=GENE):
    d = tidy[(tidy["gene_name"] == gene) & (tidy["output_type"].astype(str).str.contains("RNA_SEQ"))]
    # TODO 2: group by variant_id and take the mean of raw_score
    result = ...
    return result

In [ ]:
# CHECK 2
try:
    pe = per_variant_expression(tidy)
    assert len(pe) > 0, f"no {GENE} RNA_SEQ rows found: print tidy['output_type'].unique() and tidy['gene_name'].unique()"
    assert len(pe) <= len(variants), "more rows than variants: group by variant_id"
    print(f"OK. {len(pe)} variants have a predicted {GENE} expression change")
except (AttributeError, TypeError) as e:
    raise AssertionError("TODO 2 not finished: use d.groupby('variant_id')['raw_score'].mean()") from e

### The measured side: RNA of the cell lines that carry each variant

For each variant we take the **cell lines that carry it** and express their TP53 RNA as a **z-score** relative to the cell lines **without any TP53 mutation** in the table (how many standard deviations above or below). Negative means lower RNA than the unmutated lines.

In [ ]:
mutated_lines = set(muts["cell_line"])
ref_lines = [c for c in rna.index if c not in mutated_lines and pd.notna(rna.loc[c, GENE])]
mu, sd = rna.loc[ref_lines, GENE].mean(), rna.loc[ref_lines, GENE].std()
print(f"reference: {len(ref_lines)} cell lines without a {GENE} mutation; mean {mu:.2f}, sd {sd:.2f}")

def observed_z(carriers):
    vals = [rna.loc[c, GENE] for c in carriers if c in rna.index and pd.notna(rna.loc[c, GENE])]
    return float((np.mean(vals) - mu) / sd) if vals else np.nan

key_by_id = {f"{v.chromosome}:{v.position}:{v.reference_bases}>{v.alternate_bases}": i for i, v in enumerate(variants)}
res = pd.DataFrame({"expr_change": pe, "splice_score": per_variant_splice(tidy)})
res["i"] = [key_by_id.get(i) for i in res.index]
res = res.dropna(subset=["i"]).astype({"i": int})
res["class"] = picked.loc[res["i"], "cls"].values
res["carriers"] = picked.loc[res["i"], "n_rna"].values
res["observed_rna_z"] = [observed_z(picked.loc[i, "carriers"]) for i in res["i"]]
res = res.dropna(subset=["observed_rna_z"])
print(f"{len(res)} variants with a prediction and a measurement")
res.groupby("class")[["splice_score", "expr_change", "observed_rna_z"]].median().round(3)

In [ ]:
order = [c for c in CLASSES if (res["class"] == c).any()]
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
for a, col, lab in zip(ax[:2], ["splice_score", "expr_change"], ["max |splice-site score|", "predicted TP53 expression change"]):
    data = [res.loc[res["class"] == c, col].values for c in order]
    a.boxplot(data); a.set_xticklabels(order); a.set_ylabel(lab); a.tick_params(axis="x", rotation=30)
ax[2].scatter(res["expr_change"], res["observed_rna_z"], c=[order.index(c) for c in res["class"]], cmap="viridis", s=30)
ax[2].set_xlabel("predicted expression change"); ax[2].set_ylabel("measured TP53 RNA (z)"); ax[2].axhline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()
if len(res) >= 6:
    r_ = spearmanr(res["expr_change"], res["observed_rna_z"])
    print(f"Spearman, predicted expression change vs measured RNA z: {r_.statistic:+.2f} (p = {r_.pvalue:.2f}, n = {len(res)})")
    r2 = spearmanr(res["splice_score"], -res["observed_rna_z"])
    print(f"Spearman, splice score vs how much LOWER the measured RNA is: {r2.statistic:+.2f} (p = {r2.pvalue:.2f})")

**Debrief.**

- Was your ranking right? Which classes lowered the **measured** RNA the most? Which did AlphaGenome rank highest? Where do the two disagree?
- A **nonsense** mutation lowers RNA in the cell through **nonsense-mediated decay**, a quality-control step after transcription. Can a model that reads only DNA sequence know about that? What does this say about the question "can the model predict the effect of a mutation?" for different kinds of effect?
- Each class has at most 15 variants, and each is measured in only a few cell lines whose RNA also depends on the tissue and on every other mutation. How much would you trust a difference between classes? What would you add?
- Look at `silent_near_splice`: its median **splice score** is high, although the protein does not change. Why did we not put these variants in `silent`? What does it say about calling a variant "silent" from its protein effect alone, and where would you expect its measured RNA to sit?
- AlphaGenome's outputs must **not** be used to train other models (terms of use). Name one thing in this notebook that would break that rule if you did it next.

## Part 4. Pitfalls and open time (6 min and beyond)

**Genome build.** AlphaGenome and the CCLE table use **hg38**. Many older datasets (including most 2018-era papers) use **hg19**. The same base has a different coordinate in each. Let's see how big the shift is for one of your variants.

In [ ]:
try:
    from pyliftover import LiftOver
    lo = LiftOver("hg38", "hg19")
    v0 = variants[0]
    hit = lo.convert_coordinate(v0.chromosome, v0.position - 1)          # pyliftover takes 0-based
    pos19 = hit[0][1] + 1 if hit else None
    print(f"{v0.chromosome}:{v0.position} (hg38)  ->  {v0.chromosome}:{pos19} (hg19);  shift = {abs(v0.position - pos19):,} bases" if pos19 else "no liftover result")
except Exception as e:
    print("liftover not available here:", type(e).__name__, str(e)[:120])

**What would happen** if you passed hg19 coordinates to a tool that expects hg38? (Write your answer.) Why is the **reference-base check** from Part 2 a good safety net, and what does it *not* catch? (Hint: a variant on the wrong strand but with a matching base.)

### Challenges (pick one)

**A. More TP53 variants.** Raise `N_PER_CLASS` to 25 (watch the API quota). Does the picture sharpen?

**B. Another gene.** Set `GENE = "PTEN"` or `"KRAS"` at the top of Part 1 and re-run from there. Do the classes behave the same way?

**C. Compare the two scorers.** Repeat Part 3 with `SPLICE_SITE_USAGE` instead of `SPLICE_SITES`. Do they rank the variants the same way?

**D. Alternate allele.** For one splice variant, score all three possible alternate bases. Do they differ?

**E. The cell line, not the variant.** Cell lines differ in tissue. Colour the third plot by the tissue of the carrying line (the suffix of the cell line name). Does the measured RNA depend on tissue more than on the variant class?

## Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `API key not valid` | key copied wrongly, or secret value is the name | Re-copy the key into the secret's Value box (no quotes or spaces), or paste it when asked |
| `SecretNotFoundError` / the notebook asks for a key | secret name or access switch | Name must be `ALPHA_GENOME_API_KEY`; switch on Notebook access; or just paste the key when asked |
| Quota or `RESOURCE_EXHAUSTED` errors | many participants at once | Wait a minute and re-run; lower `N_PER_CLASS`; ask the instructor |
| Ensembl `429` or timeout | rate limit | Re-run the cell; the lookup is cached |
| `Most reference bases do not match hg38` | wrong build or shifted positions | Check that the file is the hg38 class file; do not score these variants |
| A class has no variants | the mutation types are named differently | Print `muts["variant_info"].value_counts()` and adapt `variant_class` |
| `no ... RNA_SEQ rows` | column values differ | Print `tidy['output_type'].unique()` and `tidy['gene_name'].unique()` |
| `... not finished` messages | a TODO still has `...` | Fill it in, rerun the TODO cell, then the check |
| Variable not defined | skipped a cell | Run all cells above (Runtime > Run before) |